# Beginner: Flag Potentially Biased or Exclusionary Language

**Sub-topic 1.5 — Responsible and Ethical AI**

This is a simplified version of a real technique used in responsible-AI tooling: scanning AI-assisted or AI-generated content for language patterns known to discourage certain groups of readers/applicants, before that content ships. Real tools use much larger, better-researched term lists and often ML classifiers — this keyword version is for building intuition about *what* the technique is checking for and *why*.

In [1]:
# Each category maps a flagged term to a brief note on why it's flagged
# and a more neutral alternative. This list is illustrative, not exhaustive --
# real-world bias-linting tools maintain much larger, evolving term lists.
FLAGGED_TERMS = {
    # Gender-coded language
    "ninja": ("gender-coded (masculine-coded)", "skilled / expert"),
    "rockstar": ("gender-coded (masculine-coded)", "high performer"),
    "he will": ("gendered pronoun assumption", "they will"),
    "guys": ("gendered group term", "everyone / team"),
    # Age-coded language
    "young and energetic": ("age discrimination risk", "motivated"),
    "digital native": ("age discrimination risk", "comfortable with technology"),
    "recent graduate": ("age discrimination risk", "early-career candidate"),
    # Ableist language
    "must be able to stand for long periods": ("ableist -- excludes without justification", "role requires prolonged standing; accommodations available"),
    "crazy fast": ("ableist / mental-health-stigmatizing", "very fast"),
    # Exclusionary culture-fit language
    "culture fit": ("vague, can mask bias", "culture ADD (what new perspective they bring)"),
    "work hard play hard": ("can signal poor work-life balance expectations", "collaborative, results-driven team"),
}

SAMPLE_JOB_POSTINGS = [
    (
        "Senior Engineer",
        "We're looking for a coding ninja who is a real rockstar developer. "
        "He will own our backend systems and thrive in a work hard play hard culture.",
    ),
    (
        "Marketing Associate",
        "Ideal candidate is a young and energetic digital native who is a "
        "great culture fit for our growing team.",
    ),
    (
        "Warehouse Associate",
        "Candidates must be able to stand for long periods and lift up to 50 lbs. "
        "We move crazy fast here, so hit the ground running.",
    ),
    (
        "Data Analyst",
        "We're seeking a detail-oriented analyst with strong SQL skills to join "
        "our collaborative team. Recent graduates are encouraged to apply.",
    ),
]

## The scanner

In [2]:
def scan_for_bias(text: str) -> list[tuple[str, str, str]]:
    """Return a list of (flagged_term, reason, suggested_alternative) for
    every flagged term found in `text` (case-insensitive)."""
    text_lower = text.lower()
    findings = []
    for term, (reason, alternative) in FLAGGED_TERMS.items():
        if term in text_lower:
            findings.append((term, reason, alternative))
    return findings

In [3]:
total_findings = 0
for title, posting_text in SAMPLE_JOB_POSTINGS:
    print(f"\nPosting: {title}")
    print(f"  Text: {posting_text!r}")

    findings = scan_for_bias(posting_text)
    if not findings:
        print("  No flagged terms found.")
        continue

    total_findings += len(findings)
    print(f"  Found {len(findings)} flagged term(s):")
    for term, reason, alternative in findings:
        print(f"    - {term!r}  [{reason}]  -> consider: {alternative!r}")

print(f"\n\nTotal flagged terms across all postings: {total_findings}")


Posting: Senior Engineer
  Text: "We're looking for a coding ninja who is a real rockstar developer. He will own our backend systems and thrive in a work hard play hard culture."
  Found 4 flagged term(s):
    - 'ninja'  [gender-coded (masculine-coded)]  -> consider: 'skilled / expert'
    - 'rockstar'  [gender-coded (masculine-coded)]  -> consider: 'high performer'
    - 'he will'  [gendered pronoun assumption]  -> consider: 'they will'
    - 'work hard play hard'  [can signal poor work-life balance expectations]  -> consider: 'collaborative, results-driven team'

Posting: Marketing Associate
  Text: 'Ideal candidate is a young and energetic digital native who is a great culture fit for our growing team.'
  Found 3 flagged term(s):
    - 'young and energetic'  [age discrimination risk]  -> consider: 'motivated'
    - 'digital native'  [age discrimination risk]  -> consider: 'comfortable with technology'
    - 'culture fit'  [vague, can mask bias]  -> consider: 'culture ADD (what new 

### Notice
None of these terms are automatically discriminatory in isolation — this is a **heuristic screen** meant to prompt human review, not an automatic rejection. That distinction (flag-for-review vs. auto-reject) matters enormously in responsible AI system design, and you'll see it again in Track 4's governance material.

### Try it yourself
Add 3 more biased phrases to `FLAGGED_TERMS` (research a couple of real examples of biased job-posting language) and re-run the scan.